In [4]:
import json
import os
METRICS = ['train_hard_loss', 'train_soft_loss', 'train_acc','test_loss', 'test_acc']
def get_metrics(path):
    if not os.path.exists(path):
        # print(f"Error: File not found at path '{path}'.")
        return None
    with open(path, 'r') as f:
        data = json.load(f)
    return tuple(data[metric] for metric in METRICS)

def populate_dict(path):
    if get_metrics(path) == None:
        return None
    return dict(zip(METRICS, get_metrics(path)))

In [5]:
import numpy as np

RUNS = 5 
NOISE_STDS = np.arange(0, 1.667, 0.333)
TARGETS = ['student', 'teacher', 'both']
DATASETS = ['TinyImageNet', 'Cifar100']

things = {}
for dataset in DATASETS:
    things[dataset] = {}
    for target in TARGETS:
        things[dataset][target] = {}
        for noise_std in NOISE_STDS:
            things[dataset][target][noise_std] = {}
            for run in range(RUNS):
                path = f'{dataset}/{target}/std{noise_std:.2f}/{run}/metrics.json'
                if populate_dict(path) == None:
                    continue
                things[dataset][target][noise_std][run] = populate_dict(path)


In [ ]:
def visualize_structure(things):
    def recurse(level, indent=0):
        if isinstance(level, dict):
            for key in level:
                print('  ' * indent + f"- {key}")
                recurse(level[key], indent + 1)
        else:
            print('  ' * indent + f"- {type(level).__name__}")  # or just `- value`

    print("Structure of 'things':")
    recurse(things)

In [ ]:
TARGETS = ['both', 'student', 'teacher']

for target in TARGETS:
    things[target] = {}
    std = '0.00'
    things[target][std] = {}
    for run in range(RUNS):
        path = f"experiments2/{target}/std{std}/{run}/metrics.json"


In [ ]:
thing_p = {}
for target in TARGETS:
    thing_p[target] = {}
    for metric in ['train_acc', 'test_acc']:
        thing_p[target][metric] = {}
        for run in range(RUNS):
            max_accuracy = np.array(things[target]['0.00'][run][metric])[-1]
            # max_accuracy = np.array(things[target]['0.00'][run][metric]).max()
            thing_p[target][metric][run] = max_accuracy
    
for target in TARGETS:
    for metric in ['train_acc', 'test_acc']:
        temp = []
        for run in range(RUNS):
            temp.append(thing_p[target][metric][run])
        temp = np.array(temp)
        print(f'{target}, {metric} - Mean: {temp.mean():.2f}, std: {temp.std():.2f}')


## WHY THEY WERE DIFFERENT?
## I COMPUTED THE MEAN FOR EVER EPOCH AND THEN TOOK THE MAX